# LeetCode 1446 — Consecutive Characters bằng Python

**Link đề:** https://leetcode.com/problems/consecutive-characters/
**Độ khó:** Easy · **Chủ đề:** String

```python
class Solution:
    def maxPower(self, s: str) -> int:
        ...
```

### Mục tiêu học
1. Hiểu khái niệm **substring** (chuỗi con liên tiếp) và **run** (đoạn ký tự giống nhau liên tiếp).
2. Nắm kỹ thuật **đếm độ dài run hiện tại + giữ max** trong một lượt duyệt — nền tảng của rất nhiều bài chuỗi/mảng.
3. Đi qua 5 lời giải: brute force mở rộng từ mỗi vị trí → mảng `power[i]` (theo hint) → biến đếm `O(1)` → hai con trỏ nhảy cụm → `itertools.groupby`.
4. Trace từng bước, chứng minh bằng invariant, phân tích Time/Space.
5. Luyện biến thể: run dài nhất của **một** ký tự cho trước, nén chuỗi (run-length encoding), *Max Consecutive Ones*.

**Cách học:** đọc markdown → đoán output cell code kế tiếp → chạy → so sánh. Cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

**Power** của chuỗi là độ dài lớn nhất của một **substring khác rỗng chỉ chứa một loại ký tự**. Cho chuỗi `s`, trả về power của `s`.

| Ví dụ | `s` | Output | Vì sao |
|---|---|---:|---|
| 1 | `"leetcode"` | 2 | `"ee"` |
| 2 | `"abbcccddddeeeeedcba"` | 5 | `"eeeee"` |

**Constraints**
- `1 <= s.length <= 500`
- `s` chỉ gồm chữ cái tiếng Anh viết thường.

Câu hỏi gợi mở:
- "Substring" khác "đếm số lần xuất hiện" thế nào? Trong `"abab"` chữ `a` xuất hiện 2 lần nhưng power chỉ là 1 — vì sao?
- Chuỗi dài 1 thì đáp án là bao nhiêu? (Đề đảm bảo `s` không rỗng.)

In [1]:
from collections import Counter
s = "abab"
print("Số lần xuất hiện:", Counter(s))
print("Nhưng không có 2 chữ giống nhau nào ĐỨNG CẠNH NHAU → power = 1")

Số lần xuất hiện: Counter({'a': 2, 'b': 2})
Nhưng không có 2 chữ giống nhau nào ĐỨNG CẠNH NHAU → power = 1


Nhầm "power" với "tần suất" là lỗi rất phổ biến. Power đòi các ký tự giống nhau phải **liền nhau**.

## 2. Nền tảng Python cần dùng

### 2.1. Chuỗi, chỉ số và substring

- `s[i]` là ký tự thứ `i` (đếm từ 0); `len(s)` là độ dài.
- `s[i:j]` là substring từ `i` **đến trước** `j` (độ dài `j - i`).
- So sánh ký tự: `s[i] == s[i-1]` — đây là phép so sánh ta dùng suốt bài.

In [2]:
s = "abbccc"
print(s[1], s[2], s[1] == s[2])   # b b True
print(s[3:6], len(s[3:6]))        # ccc 3
print(s[0:1])                     # a

b b True
ccc 3
a


### 2.2. Run — đoạn ký tự giống nhau liên tiếp

Chia `"abbcccddddeeeeedcba"` thành các **run**: `a | bb | ccc | dddd | eeeee | d | c | b | a`. Power = độ dài run dài nhất. Mọi substring "chỉ một loại ký tự" đều nằm **gọn trong một run**, nên chỉ cần xét các run.

In [3]:
s = "abbcccddddeeeeedcba"
runs = []
start = 0
for i in range(1, len(s) + 1):
    if i == len(s) or s[i] != s[i - 1]:   # run kết thúc trước vị trí i
        runs.append(s[start:i])
        start = i
print(runs)
print([len(r) for r in runs], "→ max =", max(len(r) for r in runs))

['a', 'bb', 'ccc', 'dddd', 'eeeee', 'd', 'c', 'b', 'a']
[1, 2, 3, 4, 5, 1, 1, 1, 1] → max = 5


Cell trên dùng điều kiện `i == len(s) or s[i] != s[i - 1]`: toán tử `or` **dừng sớm** (short-circuit) — nếu `i == len(s)` đúng thì Python **không** đánh giá `s[i]` nữa, nhờ vậy không bị `IndexError`.

### 2.3. `max` với hai đối số và cập nhật kỷ lục

`best = max(best, cur)` giữ `best` luôn là giá trị lớn nhất đã thấy. Đây là mẫu "giữ kỷ lục".

In [4]:
best = 0
for cur in [1, 2, 1, 3, 2]:
    best = max(best, cur)
    print("cur =", cur, "→ best =", best)

cur = 1 → best = 1
cur = 2 → best = 2
cur = 1 → best = 2
cur = 3 → best = 3
cur = 2 → best = 3


### 2.4. `zip(s, s[1:])` — các cặp ký tự kề nhau

In [5]:
s = "leet"
print(list(zip(s, s[1:])))       # [('l','e'), ('e','e'), ('e','t')]
print([a == c for a, c in zip(s, s[1:])])

[('l', 'e'), ('e', 'e'), ('e', 't')]
[False, True, False]


### 2.5. `itertools.groupby`

`groupby(s)` gom các phần tử **liên tiếp** bằng nhau thành nhóm, sinh cặp `(key, iterator_của_nhóm)`. Chú ý: nó chỉ gom phần tử **kề nhau** — đúng định nghĩa run, khác hẳn `Counter`.

In [6]:
from itertools import groupby
for ch, group in groupby("aabccca"):
    g = list(group)
    print(ch, g, len(g))

a ['a', 'a'] 2
b ['b'] 1
c ['c', 'c', 'c'] 3
a ['a'] 1


`group` là một **iterator** (dùng được một lần). Muốn biết độ dài phải duyệt nó: `len(list(group))` hoặc `sum(1 for _ in group)`.

## 3. Cách 1 — Brute force: từ mỗi vị trí, mở rộng sang phải

Ý tưởng trực giác: thử mọi điểm bắt đầu `i`, đi sang phải chừng nào ký tự còn giống `s[i]`, ghi lại độ dài.

In [7]:
def max_power_brute(s):
    n = len(s)
    best = 0
    for i in range(n):
        j = i
        while j < n and s[j] == s[i]:
            j += 1
        best = max(best, j - i)         # s[i:j] là run bắt đầu ở i
    return best

print(max_power_brute("leetcode"))              # 2
print(max_power_brute("abbcccddddeeeeedcba"))   # 5
print(max_power_brute("a"))                     # 1

2
5
1


### 3.1. Chi phí của brute force

Với chuỗi toàn một ký tự `"aaaa...a"` (độ dài `n`), từ vị trí `i` phải đi `n - i` bước → tổng `n + (n-1) + ... + 1 = n(n+1)/2` → `O(n²)`. Với `n = 500` vẫn ổn (~125.000), nhưng ta đang **đếm lại** cùng một run nhiều lần.

In [8]:
def count_steps_brute(s):
    n, steps = len(s), 0
    for i in range(n):
        j = i
        while j < n and s[j] == s[i]:
            j += 1
            steps += 1
    return steps

for n in [10, 100, 500]:
    print(f"n={n:>3}: 'a'*n tốn {count_steps_brute('a' * n)} bước, n(n+1)/2 = {n*(n+1)//2}")

n= 10: 'a'*n tốn 55 bước, n(n+1)/2 = 55
n=100: 'a'*n tốn 5050 bước, n(n+1)/2 = 5050
n=500: 'a'*n tốn 125250 bước, n(n+1)/2 = 125250


## 4. Nút thắt và insight

**Nút thắt:** khi bắt đầu từ `i = 1` trong `"aaaa"`, ta đếm lại `"aaa"` dù vừa đếm `"aaaa"` từ `i = 0`. Run dài `L` bị đếm `L + (L-1) + ... + 1` lần.

**Insight:** độ dài run **kết thúc tại** vị trí `i` chỉ phụ thuộc vào vị trí `i - 1`:
- nếu `s[i] == s[i-1]` → `run(i) = run(i-1) + 1`;
- ngược lại → `run(i) = 1` (bắt đầu run mới).

Đáp án = `max(run(i))`. Đây chính là hint của LeetCode: *"Giữ mảng `power[i]` là power lớn nhất kết thúc tại ký tự thứ i."*

Tính tay trên `"leetcode"`:

In [9]:
s = "leetcode"
run = 1
print(f"i=0 '{s[0]}' run=1")
for i in range(1, len(s)):
    run = run + 1 if s[i] == s[i - 1] else 1
    print(f"i={i} '{s[i]}' (trước là '{s[i-1]}') run={run}")

i=0 'l' run=1
i=1 'e' (trước là 'l') run=1
i=2 'e' (trước là 'e') run=2
i=3 't' (trước là 'e') run=1
i=4 'c' (trước là 't') run=1
i=5 'o' (trước là 'c') run=1
i=6 'd' (trước là 'o') run=1
i=7 'e' (trước là 'd') run=1


`a if điều_kiện else b` là **biểu thức điều kiện** (ternary): trả `a` nếu điều kiện đúng, ngược lại trả `b`.

## 5. Cách 2 — Mảng `power[i]` (theo hint)

Lưu toàn bộ `power[i]` vào một list rồi lấy `max`. Dễ quan sát nhưng tốn `O(n)` bộ nhớ.

In [10]:
def max_power_dp(s):
    n = len(s)
    power = [1] * n                     # mỗi ký tự tự nó là run dài 1
    for i in range(1, n):
        if s[i] == s[i - 1]:
            power[i] = power[i - 1] + 1
    return max(power)

s = "abbcccddddeeeeedcba"
print(max_power_dp(s))
power = [1] * len(s)
for i in range(1, len(s)):
    if s[i] == s[i - 1]:
        power[i] = power[i - 1] + 1
print(list(s))
print(power)

5
['a', 'b', 'b', 'c', 'c', 'c', 'd', 'd', 'd', 'd', 'e', 'e', 'e', 'e', 'e', 'd', 'c', 'b', 'a']
[1, 1, 2, 1, 2, 3, 1, 2, 3, 4, 1, 2, 3, 4, 5, 1, 1, 1, 1]


`[1] * n` tạo list gồm `n` số 1. (An toàn với số nguyên; với list lồng `[[0]] * n` thì cẩn thận vì các phần tử trỏ cùng một list.)

## 6. Cách 3 (lời giải chính) — Hai biến `cur` và `best`

Nhận xét: để tính `power[i]` ta chỉ cần `power[i-1]`. Vậy thay mảng bằng **một biến** `cur` (độ dài run hiện tại), và `best` (kỷ lục).

### 6.1. Mổ xẻ từng đoạn

**Khởi tạo:** `cur = best = 1` — vì `s` không rỗng, ký tự đầu là một run dài 1.

**Thân vòng (i từ 1):**
```python
if s[i] == s[i - 1]:
    cur += 1          # nối dài run
else:
    cur = 1           # run mới bắt đầu ở i
best = max(best, cur)
```

**Kết thúc:** `return best`.

Vì sao cập nhật `best` **mỗi vòng** (không chỉ khi run kết thúc)? Vì run dài nhất có thể nằm ở **cuối chuỗi** — nếu chỉ cập nhật ở nhánh `else` thì run cuối không bao giờ được ghi nhận (xem mục 14).

In [11]:
def max_power(s):
    cur = best = 1
    for i in range(1, len(s)):
        if s[i] == s[i - 1]:
            cur += 1
        else:
            cur = 1
        best = max(best, cur)
    return best

print(max_power("leetcode"), max_power("abbcccddddeeeeedcba"), max_power("a"), max_power("zzz"))

2 5 1 3


## 7. TRACE từng bước — `s = "abbcccd"`

| i | `s[i-1]` | `s[i]` | giống? | `cur` sau | `best` sau |
|---:|---|---|---|---:|---:|
| (khởi tạo) | – | a | – | 1 | 1 |
| 1 | a | b | không | 1 | 1 |
| 2 | b | b | có | 2 | 2 |
| 3 | b | c | không | 1 | 2 |
| 4 | c | c | có | 2 | 2 |
| 5 | c | c | có | 3 | **3** |
| 6 | c | d | không | 1 | 3 |

Kết quả: `3` (run `"ccc"`).

In [12]:
def trace_max_power(s):
    cur = best = 1
    print(f"khởi tạo: s[0]={s[0]!r} cur=1 best=1")
    for i in range(1, len(s)):
        same = s[i] == s[i - 1]
        cur = cur + 1 if same else 1
        old_best = best
        best = max(best, cur)
        note = " ← kỷ lục mới" if best > old_best else ""
        print(f"i={i}: {s[i-1]!r}->{s[i]!r} giống={same!s:5} cur={cur} best={best}{note}")
    return best

print("Kết quả:", trace_max_power("abbcccd"))

khởi tạo: s[0]='a' cur=1 best=1
i=1: 'a'->'b' giống=False cur=1 best=1
i=2: 'b'->'b' giống=True  cur=2 best=2 ← kỷ lục mới
i=3: 'b'->'c' giống=False cur=1 best=2
i=4: 'c'->'c' giống=True  cur=2 best=2
i=5: 'c'->'c' giống=True  cur=3 best=3 ← kỷ lục mới
i=6: 'c'->'d' giống=False cur=1 best=3
Kết quả: 3


### 7.1. Trace nhánh đặc biệt — run dài nhất nằm ở **cuối** chuỗi

In [13]:
print("Kết quả:", trace_max_power("abccc"))

khởi tạo: s[0]='a' cur=1 best=1
i=1: 'a'->'b' giống=False cur=1 best=1
i=2: 'b'->'c' giống=False cur=1 best=1
i=3: 'c'->'c' giống=True  cur=2 best=2 ← kỷ lục mới
i=4: 'c'->'c' giống=True  cur=3 best=3 ← kỷ lục mới
Kết quả: 3


Kỷ lục mới xuất hiện ở `i = 4` — vòng cuối cùng. Nhờ cập nhật `best` ở mỗi vòng nên không bị mất.

## 8. Cách 4 — Hai con trỏ "nhảy cụm"

Cải tiến trực tiếp từ brute force: sau khi đếm xong run `s[i:j]`, **nhảy** `i = j` thay vì `i += 1`. Mỗi ký tự chỉ được `j` đi qua một lần → `O(n)`.

In [14]:
def max_power_jump(s):
    n, i, best = len(s), 0, 0
    while i < n:
        j = i
        while j < n and s[j] == s[i]:
            j += 1
        best = max(best, j - i)
        i = j                      # bỏ qua cả run vừa đếm
    return best

print(max_power_jump("leetcode"), max_power_jump("abbcccddddeeeeedcba"))
print("Số bước với 'a'*500:", end=" ")
steps, i, s = 0, 0, "a" * 500
while i < len(s):
    j = i
    while j < len(s) and s[j] == s[i]:
        j += 1; steps += 1
    i = j
print(steps)

2 5
Số bước với 'a'*500: 500


Chỉ 500 bước thay vì 125.250 của brute force — chỉ khác một dòng `i = j`.

## 9. Cách 5 — `itertools.groupby` một dòng

In [15]:
from itertools import groupby

def max_power_groupby(s):
    return max(len(list(g)) for _, g in groupby(s))

print(max_power_groupby("leetcode"), max_power_groupby("abbcccddddeeeeedcba"))

2 5


Ngắn gọn và "rất Python", nhưng khi phỏng vấn nên giải thích được cách 3 vì người hỏi muốn xem bạn tự quản lý trạng thái.

## 10. Code nộp LeetCode (Python 3)

In [16]:
class Solution:
    def maxPower(self, s: str) -> int:
        cur = best = 1
        for i in range(1, len(s)):
            if s[i] == s[i - 1]:
                cur += 1
                if cur > best:
                    best = cur
            else:
                cur = 1
        return best

sol = Solution()
print(sol.maxPower("leetcode"))              # 2
print(sol.maxPower("abbcccddddeeeeedcba"))   # 5

2
5


Bản nộp đặt việc cập nhật `best` **trong nhánh `cur += 1`** — vẫn đúng, vì `best` chỉ có thể tăng khi `cur` tăng; khi `cur = 1` thì `best >= 1` sẵn rồi. Đây là cách viết khác của cùng ý "cập nhật mỗi khi run dài thêm" (không phải "chỉ khi run kết thúc").

## 11. Vì sao thuật toán đúng?

**Invariant** (sau khi xử lý xong chỉ số `i`):
1. `cur` = độ dài run **kết thúc tại** `i` (số ký tự giống `s[i]` liên tiếp kết thúc ở `i`).
2. `best` = độ dài run dài nhất trong `s[0..i]`.

- **Khởi tạo** (`i = 0`): run kết thúc ở 0 dài 1; `cur = best = 1`. ✓
- **Duy trì:** nếu `s[i] == s[i-1]`, run kết thúc ở `i` là run kết thúc ở `i-1` nối thêm `s[i]` → `cur + 1`. Nếu khác, run kết thúc ở `i` chỉ có `s[i]` → `1`. Mọi run trong `s[0..i]` hoặc nằm trong `s[0..i-1]` (đã có trong `best` cũ) hoặc kết thúc ở `i` (là `cur` mới, hoặc ngắn hơn) → `max(best, cur)` đúng.
- **Kết thúc:** `i = n-1` → `best` là run dài nhất cả chuỗi = power. ✓

Mọi substring chỉ một loại ký tự đều nằm trong một run, và run tự nó là substring như vậy → power = run dài nhất.

## 12. Phân tích Time / Space

Gọi `n = len(s)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute mở rộng | `O(n²)` tệ nhất | `O(1)` | đếm lại run |
| 2. Mảng `power` | `O(n)` | `O(n)` | đúng hint |
| 3. `cur` / `best` | `O(n)` | `O(1)` | **khuyên dùng** |
| 4. Nhảy cụm | `O(n)` | `O(1)` | brute force + `i = j` |
| 5. `groupby` | `O(n)` | `O(L)` cho nhóm dài nhất (`list(g)`) | ngắn nhất |

## 13. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")

sol = Solution()
check("ví dụ 1", sol.maxPower("leetcode"), 2)
check("ví dụ 2", sol.maxPower("abbcccddddeeeeedcba"), 5)
check("một ký tự", sol.maxPower("a"), 1)
check("toàn một ký tự", sol.maxPower("z" * 500), 500)
check("không có run > 1", sol.maxPower("abcdef"), 1)
check("run dài nhất ở cuối", sol.maxPower("abccc"), 3)
check("run dài nhất ở đầu", sol.maxPower("aaab"), 3)
check("tần suất cao nhưng không liền", sol.maxPower("ababab"), 1)
check("hai run cùng ký tự bị ngắt", sol.maxPower("aabaa"), 2)

PASS  ví dụ 1: 2
PASS  ví dụ 2: 5
PASS  một ký tự: 1
PASS  toàn một ký tự: 500
PASS  không có run > 1: 1
PASS  run dài nhất ở cuối: 3
PASS  run dài nhất ở đầu: 3
PASS  tần suất cao nhưng không liền: 1
PASS  hai run cùng ký tự bị ngắt: 2


### 13.1. Đối chiếu 5 cách — ngẫu nhiên có seed

Dùng bảng chữ nhỏ (`"ab"`, `"abc"`) để tạo nhiều run dài.

In [18]:
import random
rng = random.Random(1446)
methods = [max_power_brute, max_power_dp, max_power, max_power_jump, max_power_groupby, Solution().maxPower]
for _ in range(1000):
    alpha = rng.choice(["a", "ab", "abc", "abcdefghijklmnopqrstuvwxyz"])
    s = "".join(rng.choice(alpha) for _ in range(rng.randint(1, 60)))
    results = {f(s) for f in methods}
    assert len(results) == 1, (s, results)
print("1000 chuỗi ngẫu nhiên: 6 hàm đều cho cùng kết quả")

1000 chuỗi ngẫu nhiên: 6 hàm đều cho cùng kết quả


`{f(s) for f in methods}` là **set** các kết quả; nếu mọi hàm đồng ý thì set chỉ có 1 phần tử — một mẹo gọn để so nhiều hàm cùng lúc.

## 14. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ cập nhật `best` khi run kết thúc

```python
if s[i] == s[i-1]:
    cur += 1
else:
    best = max(best, cur)   # chỉ ghi khi đổi ký tự
    cur = 1
return best                 # quên run cuối!
```

In [19]:
def wrong_update_on_break(s):
    cur = best = 1
    for i in range(1, len(s)):
        if s[i] == s[i - 1]:
            cur += 1
        else:
            best = max(best, cur)
            cur = 1
    return best

print("Sai:", wrong_update_on_break("abccc"), "| Đúng:", max_power("abccc"))

Sai: 1 | Đúng: 3


**Sửa:** cập nhật mỗi vòng, hoặc thêm `return max(best, cur)` ở cuối.

### Lỗi 2 — Nhầm power với tần suất lớn nhất

In [20]:
def wrong_frequency(s):
    return max(Counter(s).values())

print("Sai:", wrong_frequency("aabaa"), "| Đúng:", max_power("aabaa"))

Sai: 4 | Đúng: 2


**Vì sao sai:** `Counter` đếm mọi lần xuất hiện, không quan tâm liền nhau.

### Lỗi 3 — Khởi tạo `best = 0`, `cur = 0` rồi duyệt từ `i = 1`

Với `s = "a"`, vòng lặp không chạy → trả `0`, nhưng đáp án là `1`. Tệ hơn, ký tự đầu không bao giờ được tính vào run đầu tiên, nên mọi run bắt đầu ở vị trí 0 bị **thiếu 1**.

In [21]:
def wrong_init_zero(s):
    cur = best = 0
    for i in range(1, len(s)):
        cur = cur + 1 if s[i] == s[i - 1] else 1
        best = max(best, cur)
    return best

print("Sai:", wrong_init_zero("a"), wrong_init_zero("aab"), "| Đúng:", max_power("a"), max_power("aab"))

Sai: 0 1 | Đúng: 1 2


Với `"aab"`: `i=1` giống → `cur = 0 + 1 = 1` (đáng lẽ 2) → thiếu 1. **Sửa:** `cur = best = 1` vì ký tự đầu đã là run dài 1.

### Lỗi 4 — So sánh với ký tự kế tiếp và vượt chỉ số

```python
for i in range(len(s)):
    if s[i] == s[i + 1]:   # IndexError ở i = len(s) - 1
```

In [22]:
try:
    s = "aab"
    for i in range(len(s)):
        _ = s[i] == s[i + 1]
except IndexError as e:
    print("Lỗi:", e)
print("Sửa: duyệt i từ 1 và so s[i] với s[i-1], hoặc range(len(s) - 1) cho s[i+1]")

Lỗi: string index out of range
Sửa: duyệt i từ 1 và so s[i] với s[i-1], hoặc range(len(s) - 1) cho s[i+1]


### Lỗi 5 — Quên `i = j` trong cách nhảy cụm → vòng lặp vô hạn hoặc chậm

Nếu viết `i += 1` thì quay về brute force `O(n²)`; nếu quên hẳn dòng cập nhật `i` thì vòng `while i < n` **không bao giờ dừng**. (Không chạy code vô hạn ở đây — chỉ ghi nhớ.)

## 15. Mẹo nhận dạng pattern

- Từ khóa "**liên tiếp / consecutive / contiguous / substring chỉ gồm…**" + chỉ cần **độ dài** → đếm run với `cur` + `best`.
- Mẫu chung: `cur = cur + 1 if (phần tử i "nối tiếp" được i-1) else (giá trị khởi đầu)`; `best = max(best, cur)`.
- Đây là dạng đơn giản nhất của **DP một chiều** (`power[i]` phụ thuộc `power[i-1]`) và thu gọn bộ nhớ thành 1 biến.
- Bài cùng họ: *Max Consecutive Ones* (485), *Longest Continuous Increasing Subsequence* (674), *Longer Contiguous Segments of Ones than Zeros* (1869), *String Compression* (443).

## 16. Biến thể

### 16.1. Run dài nhất của **một** ký tự cho trước (giống Max Consecutive Ones)

In [23]:
def longest_run_of(s, ch):
    cur = best = 0
    for c in s:
        cur = cur + 1 if c == ch else 0
        best = max(best, cur)
    return best

print(longest_run_of("abbcccddddeeeeedcba", "d"))   # 4
print(longest_run_of("abbcccddddeeeeedcba", "z"))   # 0
print(longest_run_of([1, 1, 0, 1, 1, 1], 1))        # 3 — dùng được cho list

4
0
3


Khác bài gốc: run "đứt" về `0` (không phải `1`) khi gặp ký tự khác, và có thể trả `0`.

### 16.2. Trả về cả ký tự và vị trí bắt đầu của run dài nhất

In [24]:
def longest_run_info(s):
    best_len, best_start, start = 1, 0, 0
    for i in range(1, len(s)):
        if s[i] != s[i - 1]:
            start = i
        if i - start + 1 > best_len:
            best_len, best_start = i - start + 1, start
    return s[best_start], best_start, best_len

print(longest_run_info("abbcccddddeeeeedcba"))   # ('e', 10, 5)

('e', 10, 5)


Dùng `>` (không phải `>=`) nên khi có nhiều run cùng độ dài, ta giữ run **xuất hiện sớm nhất**.

### 16.3. Run-length encoding (nén chuỗi)

In [25]:
def rle(s):
    return "".join(f"{ch}{len(list(g))}" for ch, g in groupby(s))

print(rle("aaabccdddd"))   # a3b1c2d4

a3b1c2d4


## 17. Bài tập luyện tập (tăng dần)

**Bài 1 — Tính tay.** Viết dãy `power[i]` cho `s = "aabbbcbb"`. Đáp án là gì?

**Bài 2 — Dự đoán output.**
```python
from itertools import groupby
print([k for k, _ in groupby("aabbbcbb")])
```

Bài 3–6 có cell TODO.

### Bài 3 — TODO: tự viết lời giải `cur` / `best`

In [26]:
def exercise_max_power(s):
    # TODO 1: cur = best = 1
    # TODO 2: duyệt i từ 1, so s[i] với s[i-1]
    # TODO 3: cập nhật best mỗi vòng
    pass

# assert exercise_max_power("leetcode") == 2
# assert exercise_max_power("abccc") == 3

### Bài 4 — TODO: Max Consecutive Ones (LeetCode 485)
Cho mảng nhị phân, trả độ dài đoạn số 1 liên tiếp dài nhất.

In [27]:
def exercise_max_consecutive_ones(nums):
    # TODO: run "đứt" về 0 khi gặp số 0
    pass

# assert exercise_max_consecutive_ones([1,1,0,1,1,1]) == 3
# assert exercise_max_consecutive_ones([0,0]) == 0

### Bài 5 — TODO: Longest Continuous Increasing Subsequence (LeetCode 674)
Độ dài đoạn con liên tiếp **tăng ngặt** dài nhất. Chỉ khác bài gốc ở điều kiện "nối tiếp".

In [28]:
def exercise_lcis(nums):
    # TODO: điều kiện nối tiếp là nums[i] > nums[i-1]
    pass

# assert exercise_lcis([1,3,5,4,7]) == 3
# assert exercise_lcis([2,2,2,2]) == 1

### Bài 6 — Mini challenge: power sau khi được đổi tối đa 1 ký tự
Được đổi **tối đa một** ký tự của `s` thành bất kỳ chữ cái nào. Power lớn nhất đạt được là bao nhiêu?
Ví dụ `"aabaa"` → đổi `b` thành `a` → `"aaaaa"` → 5.

In [29]:
def exercise_power_one_change(s):
    # TODO: với mỗi ký tự ch, tìm cửa sổ dài nhất chứa tối đa 1 ký tự khác ch
    #       (sliding window), lấy max qua mọi ch
    pass

# assert exercise_power_one_change("aabaa") == 5
# assert exercise_power_one_change("abc") == 2
# assert exercise_power_one_change("a") == 1

## 18. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 3 | Run kết thúc ở `i` phụ thuộc `i-1`. | 1 vòng `for i in range(1, n)`. | `cur = cur+1 if s[i]==s[i-1] else 1; best = max(best, cur)` |
| 4 | Như 16.1 với `ch = 1`. | `cur`, `best` khởi tạo 0. | `cur = cur+1 if x==1 else 0` |
| 5 | Chỉ đổi điều kiện "nối tiếp". | giống bài 3. | `if nums[i] > nums[i-1]` |
| 6 | Cửa sổ trượt: số ký tự "lạ" trong cửa sổ ≤ 1. | hai con trỏ `left`, `right`, biến `bad`. | thêm `s[right]`; nếu `bad > 1` thì tăng `left` tới khi `bad <= 1`; `best = max(best, right-left+1)` |

## 19. Đáp án tham khảo (chỉ xem sau khi tự làm)

**Bài 1:** `power = [1,2,1,2,3,1,1,2]` → `3`.
**Bài 2:** `['a', 'b', 'c', 'b']` — `groupby` chỉ gom các phần tử kề nhau nên `'b'` xuất hiện 2 lần.

In [30]:
def answer_max_power(s):
    cur = best = 1
    for i in range(1, len(s)):
        cur = cur + 1 if s[i] == s[i - 1] else 1
        best = max(best, cur)
    return best

def answer_max_consecutive_ones(nums):
    cur = best = 0
    for x in nums:
        cur = cur + 1 if x == 1 else 0
        best = max(best, cur)
    return best

def answer_lcis(nums):
    cur = best = 1
    for i in range(1, len(nums)):
        cur = cur + 1 if nums[i] > nums[i - 1] else 1
        best = max(best, cur)
    return best

def answer_power_one_change(s):
    best = 0
    for ch in set(s):
        left = bad = 0
        for right, c in enumerate(s):
            if c != ch:
                bad += 1
            while bad > 1:
                if s[left] != ch:
                    bad -= 1
                left += 1
            best = max(best, right - left + 1)
    return best

assert answer_max_power("leetcode") == 2
assert answer_max_power("abccc") == 3
assert answer_max_consecutive_ones([1,1,0,1,1,1]) == 3
assert answer_max_consecutive_ones([0,0]) == 0
assert answer_lcis([1,3,5,4,7]) == 3
assert answer_lcis([2,2,2,2]) == 1
assert answer_power_one_change("aabaa") == 5
assert answer_power_one_change("abc") == 2
assert answer_power_one_change("a") == 1
print("Đáp án bài 3–6: PASS")

Đáp án bài 3–6: PASS


In [31]:
# Kiểm chứng bài 6 bằng vét cạn (thử đổi từng vị trí thành từng chữ cái) + bài 1, 2
def brute_one_change(s):
    best = max_power(s)
    for i in range(len(s)):
        for ch in "abc":
            best = max(best, max_power(s[:i] + ch + s[i+1:]))
    return best
rng = random.Random(6)
for _ in range(300):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 15)))
    assert answer_power_one_change(s) == brute_one_change(s), s
print("Bài 6 khớp vét cạn")
s = "aabbbcbb"
power = [1] * len(s)
for i in range(1, len(s)):
    if s[i] == s[i - 1]:
        power[i] = power[i - 1] + 1
print("Bài 1:", power, max(power))
print("Bài 2:", [k for k, _ in groupby("aabbbcbb")])

Bài 6 khớp vét cạn
Bài 1: [1, 2, 1, 2, 3, 1, 1, 2] 3
Bài 2: ['a', 'b', 'c', 'b']


## 20. Template nhớ nhanh + checklist submit

```python
cur = best = GIÁ_TRỊ_CHO_PHẦN_TỬ_ĐẦU      # thường là 1
for i in range(1, n):
    if NỐI_TIẾP_ĐƯỢC(i - 1, i):
        cur += 1
    else:
        cur = GIÁ_TRỊ_KHỞI_ĐẦU             # 1 (hoặc 0 nếu phần tử i không thuộc run)
    best = max(best, cur)                  # cập nhật MỖI vòng
return best
```

**Checklist trước khi nộp**
- [ ] Power là run **liền nhau**, không phải tần suất.
- [ ] `cur = best = 1` vì chuỗi không rỗng.
- [ ] Cập nhật `best` cả khi run nằm ở **cuối** chuỗi.
- [ ] So `s[i]` với `s[i-1]` khi `i` chạy từ 1 (tránh `IndexError`).
- [ ] Method tên `maxPower`, trả về `int`.